# LeetCode #1146: Snapshot Array

https://leetcode.com/problems/snapshot-array/

## Comparison of Approaches

| Approach | Time Complexity | Space Complexity |
| :--- | :--- | :--- |
| **Brute Force** | $O(n)$ snap, $O(1)$ set/get | $O(n \times s)$ |
| **Optimal: Per-Index Snapshot List + Binary Search ★** | $O(\log s)$ get, $O(1)$ set/snap | $O(n + \text{writes})$ |

---

## Understanding the Methods

### Brute Force
On every `snap()`, copy the entire array of length $n$ into a new snapshot. `get(i, snap_id)` indexes directly. Correct but costs $O(n)$ time and $O(n \times s)$ space per snapshot, which is prohibitive if $n$ and $s$ are both large.

### Optimal: Per-Index Snapshot List + Binary Search ★
Each index stores only a list of `(snap_id, value)` pairs — one entry per `set()` call at that index. `get(i, snap_id)` binary-searches the list for the largest `snap_id \leq$ the requested one. Unchanged indices carry no overhead.

**Why this is better than Brute Force:** Space shrinks from $O(n \times s)$ to $O(n + \text{write count})$; `get` degrades gracefully to $O(\log s)$ while `set` and `snap` remain $O(1)$.

**Constraints:**
* $1 \leq n \leq 5 \times 10^4$
* $0 \leq \text{val} \leq 10^9$
* At most $5 \times 10^4$ calls to `set`, `snap`, and `get`


## Solutions

### C#

In [ ]:
public class SnapshotArray {
    private List<(int snapId, int val)>[] history;
    private int snapCount = 0;

    public SnapshotArray(int length) {
        history = new List<(int, int)>[length];
        for (int i = 0; i < length; i++)
            history[i] = new List<(int, int)> { (0, 0) }; // seed with snap 0, value 0
    }

    public void Set(int index, int val) {
        var h = history[index];
        // Overwrite if already recorded in the current snapshot, otherwise append
        if (h[^1].snapId == snapCount)
            h[^1] = (snapCount, val);
        else
            h.Add((snapCount, val));
    }

    public int Snap() => snapCount++;

    public int Get(int index, int snapId) {
        var h = history[index];
        // Binary search for the latest entry whose snapshot ID is <= snapId
        int lo = 0, hi = h.Count - 1;
        while (lo < hi) {
            int mid = (lo + hi + 1) / 2;
            if (h[mid].snapId <= snapId) lo = mid;
            else hi = mid - 1;
        }
        return h[lo].val;
    }
}

### Python

In [ ]:
import bisect

class SnapshotArray:
    def __init__(self, length: int):
        # Each index starts with [(snap_id=0, value=0)] to simplify binary search
        self.history = [[(0, 0)] for _ in range(length)]
        self.snap_count = 0

    def set(self, index: int, val: int) -> None:
        h = self.history[index]
        # Overwrite the current-snapshot entry to avoid redundant pairs
        if h[-1][0] == self.snap_count:
            h[-1] = (self.snap_count, val)
        else:
            h.append((self.snap_count, val))

    def snap(self) -> int:
        self.snap_count += 1
        return self.snap_count - 1

    def get(self, index: int, snap_id: int) -> int:
        h = self.history[index]
        # bisect_right on snap_ids; step back one to get the floor entry
        pos = bisect.bisect_right(h, (snap_id, float('inf'))) - 1
        return h[pos][1]

### Go

In [ ]:
import "sort"

type SnapshotArray struct {
    history [][]struct{ snapID, val int }
    snapCount int
}

func Constructor(length int) SnapshotArray {
    h := make([][]struct{ snapID, val int }, length)
    for i := range h {
        // Seed each index so binary search always has a valid floor
        h[i] = []struct{ snapID, val int }{{0, 0}}
    }
    return SnapshotArray{history: h}
}

func (s *SnapshotArray) Set(index, val int) {
    h := s.history[index]
    // Merge into existing entry for this snapshot to avoid duplicates
    if h[len(h)-1].snapID == s.snapCount {
        s.history[index][len(h)-1].val = val
    } else {
        s.history[index] = append(h, struct{ snapID, val int }{s.snapCount, val})
    }
}

func (s *SnapshotArray) Snap() int {
    id := s.snapCount
    s.snapCount++
    return id
}

func (s *SnapshotArray) Get(index, snapID int) int {
    h := s.history[index]
    // Find the rightmost entry whose snapID <= requested snapID
    pos := sort.Search(len(h), func(i int) bool { return h[i].snapID > snapID }) - 1
    return h[pos].val
}

### Rust

In [ ]:
use std::collections::HashMap;

struct SnapshotArray {
    // Each index maps to a sorted list of (snap_id, value) pairs
    history: Vec<Vec<(i32, i32)>>,
    snap_count: i32,
}

impl SnapshotArray {
    fn new(length: i32) -> Self {
        let length = length as usize;
        // Seed with snap_id=0, val=0 so binary search always has a floor
        let history = vec![vec![(0i32, 0i32)]; length];
        SnapshotArray { history, snap_count: 0 }
    }

    fn set(&mut self, index: i32, val: i32) {
        let h = &mut self.history[index as usize];
        // Overwrite the last entry if it belongs to the current snapshot
        if h.last().unwrap().0 == self.snap_count {
            *h.last_mut().unwrap() = (self.snap_count, val);
        } else {
            h.push((self.snap_count, val));
        }
    }

    fn snap(&mut self) -> i32 {
        let id = self.snap_count;
        self.snap_count += 1;
        id
    }

    fn get(&self, index: i32, snap_id: i32) -> i32 {
        let h = &self.history[index as usize];
        // Binary search for the floor entry whose snap_id <= snap_id requested
        let pos = h.partition_point(|&(sid, _)| sid <= snap_id) - 1;
        h[pos].1
    }
}

## Example Scenarios

### 1. Common Case
**Input:** `SnapshotArray(3)`, `set(0, 5)`, `snap()` → 0, `get(0, 0)` → **5**
Index 0's history after set: `[(0,5)]`. snap advances to 1. get binary-searches for floor at snap 0, finds `(0,5)` → 5.

### 2. Slightly Complex
**Input:** `set(0,6)`, `snap()` → 1, `set(0, 7)`, `get(0, 1)` → **6**
After the second set, index 0 has `[(0,5),(1,6),(2,7)]`. Searching for floor at snap_id=1 returns `(1,6)` → 6. The more-recent value 7 is correctly excluded.

### 3. Edge Case: Time Factor
**Input:** $5 \times 10^4$ calls alternating `set(0, i)` and `snap()`
Index 0 accumulates $2.5 \times 10^4$ history entries. Each `get` binary-searches $O(\log 2.5 \times 10^4) \approx 15$ steps, confirming the $O(\log s)$ bound even at maximum snap count.

### 4. Edge Case: Space Factor
**Input:** `SnapshotArray(50000)`, only `snap()` called $5 \times 10^4$ times, no `set()`
Every index retains only its seed entry `[(0,0)]`. Total storage = $n$ entries $\approx 5 \times 10^4$ pairs — $O(n)$ regardless of snap count. Confirms that snapshots without writes cost nothing per index.

### 5. Almost-Impossible but Plausible
**Input:** `set(0, 0)` repeated in the same snapshot $10^4$ times before `snap()`
Each call overwrites the same `(snap_id, val)` pair — no new entries are added. History for index 0 stays length 1. Confirms the overwrite-in-place optimisation prevents runaway memory growth from repeated sets.
